# 🛡️ DeepShield Audio — Notebook 05: Explainability (Grad-CAM)

This notebook demonstrates how to compute Gradient-weighted Class Activation Mapping (Grad-CAM) to visualize which parts of the Log-Mel Spectrogram the model is focusing on to make its "Spoof" or "Bonafide" prediction.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2

from src.models.custom_cnn import build_custom_cnn
from src.explainability import compute_gradcam, overlay_gradcam
from src.config import N_MELS, T_FRAMES

plt.rcParams['figure.facecolor'] = '#1E1E2E'
plt.rcParams['axes.facecolor'] = '#1E1E2E'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'

print('✅ Imports successful')

## 1. Load Model and Generate Dummy Input
We load a compiled CNN and simulate a spectrogram input.

In [ ]:
model = build_custom_cnn()
last_conv_layer_name = 'conv2d_3'

# Create a synthetic spectrogram containing some "patterns"
spec = np.zeros((N_MELS, T_FRAMES), dtype=np.float32)
spec[40:60, 50:100] = 5.0  # Synthetic "formant"
spec[80:100, 150:200] = 3.0 # Synthetic "noise"

# Prepare model input (batch_size, height, width, channels)
model_input = np.expand_dims(np.expand_dims(spec, axis=-1), axis=0)
print(f"Input shape: {model_input.shape}")

## 2. Compute Grad-CAM Heatmap
The `compute_gradcam` function traces the gradients from the output prediction back to the specified convolutional layer to see what activated it.

In [ ]:
heatmap = compute_gradcam(model, model_input, last_conv_layer_name)

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(heatmap, cmap='jet', aspect='auto', origin='lower')
plt.colorbar(im)
ax.set_title('Raw Grad-CAM Heatmap (Low Res)')
plt.show()

## 3. Overlay Heatmap on Original Spectrogram
To make it interpretable, we upscale the heatmap to the original spectrogram size and overlay it.

In [ ]:
# Resize heatmap to original shape
heatmap_resized = cv2.resize(heatmap, (spec.shape[1], spec.shape[0]))
heatmap_resized = np.uint8(255 * heatmap_resized)
heatmap_colored = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)

# Normalize spec for visualization overlay
spec_norm = (spec - spec.min()) / (spec.max() - spec.min() + 1e-8)
spec_rgb = np.stack([spec_norm]*3, axis=-1)
spec_rgb = np.uint8(255 * spec_rgb)

# Superimpose
superimposed_img = heatmap_colored * 0.4 + spec_rgb * 0.6
superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].imshow(spec, origin='lower', aspect='auto', cmap='magma')
axes[0].set_title('Original Log-Mel Spectrogram')
axes[0].set_ylabel('Mel Bin')
axes[0].set_xlabel('Time Frame')

axes[1].imshow(superimposed_img, origin='lower', aspect='auto')
axes[1].set_title('Spectrogram + Grad-CAM Overlay')
axes[1].set_ylabel('Mel Bin')
axes[1].set_xlabel('Time Frame')

plt.show()
print("Notice how the heatmap highlights the regions that triggered the model's prediction!")